In [1]:
using MarineHydro
using PyCall
using DimensionalData
cpt = pyimport("capytaine")

# Description of problem
h = Inf # sea depth [m]
omegas = [1.03, 1.7] # frequencies [rad/s]
betas = [0.0, pi/6] # incident wave angle [rad]
beta = betas[1]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Capytaine Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
rotation_center = (1.0, 1.0, 0.0) # off set for nonzero off-diagoinal elements
len = 2.5
faces_max_radius = 0.5
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)



ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.


PyObject Mesh(vertices=[[... 83 vertices ...]], faces=[[... 80 faces ...]], name="cylinder_0")

In [2]:
# Solve using Capytaine

# Create FloatingBody object
cptbody = cpt.FloatingBody(mesh=cptmesh)
cptbody.center_of_mass = center
cptbody.rotation_center = rotation_center
foreach(dof -> cptbody.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody.add_rotation_dof(name=dof), r_DOFs)
cptbody.active_dofs = DOFs
cptbody.name = "Horizontal Cylinder 1"

# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptbody.active_dofs
xr = pyimport("xarray")
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => DOFs))
cptresults = cpt.BEMSolver().fill_dataset(test_matrix, cptbody, method="direct")

# Get Capytaine values
# A_cpt = cptresults.added_mass
# B_cpt = cptresults.radiation_damping
# F_FK_cpt = cptresults.Froude_Krylov_force 
# F_D_cpt = cptresults.diffraction_force
# F_ex_cpt = cptresults.excitation_force

Solving BEM problems ---------------------------------------- 100% 0:00:00


PyObject <xarray.Dataset> Size: 740B
Dimensions:              (omega: 2, radiating_dof: 2, influenced_dof: 2,
                          wave_direction: 2)
Coordinates:
    g                    float64 8B 9.81
    rho                  float64 8B 1e+03
    body_name            <U21 84B 'Horizontal Cylinder 1'
    water_depth          float64 8B inf
    forward_speed        float64 8B 0.0
  * wave_direction       (wave_direction) float64 16B 0.0 0.5236
  * omega                (omega) float64 16B 1.03 1.7
  * radiating_dof        (radiating_dof) object 16B 'Heave' 'Pitch'
  * influenced_dof       (influenced_dof) object 16B 'Heave' 'Pitch'
    period               (omega) float64 16B 6.1 3.696
    wavenumber           (omega) float64 16B 0.1081 0.2946
    wavelength           (omega) float64 16B 58.1 21.33
Data variables:
    added_mass           (omega, radiating_dof, influenced_dof) float64 64B 6...
    radiation_damping    (omega, radiating_dof, influenced_dof) float64 64B 1...
    diffraction_force    (omega, wave_direction, influenced_dof) complex128 128B ...
    Froude_Krylov_force  (omega, wave_direction, influenced_dof) complex128 128B ...
    excitation_force     (omega, wave_direction, influenced_dof) complex128 128B ...
Attributes: (12/16)
    start_of_computation:                     2026-03-26T20:02:58.128600
    green_function:                           Delhommeau
    tabulation_nr:                            676
    tabulation_rmax:                          100.0
    tabulation_nz:                            372
    tabulation_zmin:                          -251.0
    ...                                       ...
    gf_singularities:                         low_freq
    engine:                                   BasicMatrixEngine
    matrix_cache_size:                        1
    linear_solver:                            lu_decomposition
    creation_of_dataset:                      2026-03-26T20:02:58.140860
    capytaine_version:                        2.2.1

In [3]:
using Zygote
omegas = [1.0, 1.5] # frequencies [rad/s]
beta = 0 # incident wave angle [rad]
t_DOFs = ["Heave"] # translational DOFs
r_DOFs = ["Pitch"] # rotational DOFs
DOFs = [t_DOFs; r_DOFs] # all DOFs

# Create Mesh object
radius = 1.5  
center = (0.0,0.0,0.0) 
len = 2.5
faces_max_radius = 0.7
cpt = pyimport("capytaine")
cptmesh = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)

# Get MarineHydro values
mesh = Mesh(cptmesh)
rigid_dof_list = DOFs
rotation_center = [1.0, 1.0, 0.0] # off set for nonzero off-diagoinal elements
floatingbody = FloatingBody(mesh, rigid_dof_list, rotation_center, "Horizontal_Cylinder")

# Radiation solve functions
function A_and_B_vec(w)
    parameters = (wave_frequencies=w,
        radiating_dofs=collect(keys(floatingbody.dofs)),
        influenced_dofs=collect(keys(floatingbody.dofs)))
    data = compute_hydrodynamic_coefficients(parameters, floatingbody)
    return vcat(vec(data.added_mass), vec(data.radiation_damping)) 
end
function Jacobian_of_rad_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        A_and_B_vec(real(w)) 
    end
    return vec(real.(j)[1])
end

# Incident + diffraction solve function
function F_ex_vec(w)
    parameters = (wave_frequencies=[w],
        wave_directions=[beta],
        influenced_dofs=collect(keys(floatingbody.dofs)))
    data = compute_hydrodynamic_coefficients(parameters, floatingbody)
    return vcat(real.(vec(data.excitation_force)),imag.(vec(data.excitation_force))) 
end
function Jacobian_of_dif_problem(Omega)
    # This inclusion of imaginary inputs, then taking the real value was required to get jacobian to work
    j = Zygote.jacobian(Omega + 0im) do w
        F_ex_vec(real(w)) 
    end
    return vec(real.(j)[1])
end

Jacobian_of_dif_problem (generic function with 1 method)

In [4]:
F_ex_vec(1.5)

StackOverflowError: StackOverflowError:

In [6]:
Jacobian_of_dif_problem(1.5)

ErrorException: Need an adjoint for constructor DiffractionResult. Gradient is of type ChainRulesCore.Tangent{Any, @NamedTuple{problem::ChainRulesCore.Tangent{Any, @NamedTuple{floatingbody::ChainRulesCore.ZeroTangent, omega::ComplexF64, beta::Float64, influenced_dofs::ChainRulesCore.ZeroTangent}}, forces::ChainRulesCore.Tangent{Any, @NamedTuple{Heave::ComplexF64, Pitch::ComplexF64}}}}

In [3]:
# Solve using MarineHydro

# Create Mesh struct
mhmesh = Mesh(cptmesh)

# Create FloatingBody struct
floatingbody1 = FloatingBody(mhmesh,
    Symbol.(DOFs),
    [rotation_center...],
    "Horizontal Cylinder 1")

# Create parameters NamedTuple
parameters = (wave_frequencies=omegas, 
wave_directions=betas,
radiating_dofs=Symbol.(DOFs))

# Compute all hydrodynamic coefficients
mh_hydro_coeffs = compute_hydrodynamic_coefficients(parameters, floatingbody1)


(added_mass = [6821.175181729206 6821.175181729206; 6821.175181729203 9953.867547922553;;; 5313.377249586668 5313.377249586667; 5313.377249586667 8676.459451189043], radiation_damping = [1824.9195800001207 1824.91958000012; 1824.9195800001198 1833.2261657072925;;; 3674.3651819090996 3674.3651819090996; 3674.3651819090987 3945.911665710678], diffraction_force = ComplexF64[-6747.126634168553 - 1903.1266489043915im -11880.05492182946 - 6787.505009516311im; -6769.683684555786 + 1472.5471297037038im -12836.10995581038 + 2535.269233304113im;;; -6808.347186106086 - 1900.594856115948im -12278.012121686026 - 6724.27564561899im; -6827.894252619126 + 1028.5397455819598im -13110.540749016265 + 1456.8565231192556im], Froude_Krylov_force = ComplexF64[63068.81437521969 + 1.4709558352063256e-13im 49963.68175155532 + 3.4997726253514167e-13im; 63068.814375219685 + 1832.6589348041916im 49963.681751555305 + 4585.544567114047im;;; 63048.6662446628 - 4.656067979220311e-13im 49827.84542774871 - 5.60713413848

In [ ]:
parameters = (wave_frequencies=[omegas[1]],
    wave_directions=betas)
res = compute_hydrodynamic_coefficients(parameters, floatingbody1)

2×1×2 Array{ComplexF64, 3}:
[:, :, 1] =
 -6747.126634168553 - 1903.1266489043915im
 -6769.683684555786 + 1472.5471297037038im

[:, :, 2] =
 -6808.347186106086 - 1900.594856115948im
 -6827.894252619126 + 1028.5397455819598im

In [36]:
parameters.radiating_dofs

2-element Vector{Symbol}:
 :Heave
 :Pitch

In [ ]:
# Create DimStack (not differentiable)
mhresults = compute_and_label_hydrodynamic_coefficients(parameters, floatingbody1)

In [ ]:
# Compare added_mass
omega_num = 2

display("Capytaine Added Mass")
display(cptresults.added_mass.sel(omega=omegas[omega_num]).values)

display("MarineHydro Added Mass")
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[omega_num])]))

In [ ]:
# Multiple Body Case

# Create Capytaine Mesh object
radius = 1.5  
sep_dis = 50.0
center = (sep_dis,sep_dis,0.0) 
rotation_center = (sep_dis + 1.0, sep_dis + 1.0, 0.0) # off set for nonzero off-diagoinal elements
len = 2.5
faces_max_radius = 0.5
cptmesh2 = cpt.meshes.predefined.mesh_horizontal_cylinder(
            radius=radius,
            center=center, 
            length=len, 
            faces_max_radius = faces_max_radius
            ).keep_immersed_part(inplace=true)


# Create FloatingBody object
cptbody2 = cpt.FloatingBody(mesh=cptmesh2)
cptbody2.center_of_mass = center
cptbody2.rotation_center = rotation_center
foreach(dof -> cptbody2.add_translation_dof(name=dof), t_DOFs)
foreach(dof -> cptbody2.add_rotation_dof(name=dof), r_DOFs)
cptbody2.active_dofs = DOFs
cptbody2.name = "Horizontal Cylinder 2"

# Create Mesh struct
mhmesh2 = Mesh(cptmesh2)

# Create FloatingBody struct
floatingbody2 = FloatingBody(mhmesh2,
    Symbol.(DOFs),
    [rotation_center...],
    "Horizontal Cylinder 2")

floatingbodylist = [floatingbody1, floatingbody2]


In [ ]:
# Setup and solve BEM problems
cptarray = cptbody + cptbody2

# Setup and solve BEM problems
solver = cpt.BEMSolver()
dof_list = cptarray.dofs
xr = pyimport("xarray")
rad_dofs = string.(collect(keys(dof_list)))
test_matrix = xr.Dataset(coords=Dict("omega" => omegas, "wave_direction" => betas, "radiating_dof" => rad_dofs))
cptresults = cpt.BEMSolver().fill_dataset(test_matrix, cptarray, method="direct")



In [ ]:
#Create and array (just another floating body)
fb_array = combine_floatingbodies(floatingbodylist)

# Create parameters NamedTuple
parameters = (wave_frequencies=omegas, 
wave_directions=betas,
radiating_dofs=collect(keys(fb_array.dofs)))

# Create DimStack of results
mhresults = compute_and_label_hydrodynamic_coefficients(parameters, fb_array)

In [ ]:
keys(cptarray.dofs)

In [ ]:
cptresults.added_mass

In [ ]:
DOFz=String.(collect(keys(fb_array.dofs)))

In [ ]:
keys(fb_array.dofs)

In [ ]:
mhresults.added_mass[influenced_dofs = At(Symbol(DOFz[1])),radiating_dofs = At(Symbol(DOFz[2])), wave_frequencies = At(omegas[1])]

In [ ]:
# Compare added_mass
omega_num = 1

display("Capytaine Added Mass")
display(cptresults.added_mass.sel(omega=omegas[omega_num]).values)

display("MarineHydro Added Mass")
display(collect(mhresults.added_mass[wave_frequencies = At(omegas[omega_num])]))

In [ ]:
rad_dofs[1]

In [ ]:
rad_dofs[1:3]

In [ ]:
#Create and array (just another floating body)
fb_array = combine_floatingbodies(floatingbodylist)

rad_dofs = collect(keys(fb_array.dofs))

# Create parameters NamedTuple
parameters = (wave_frequencies=[omegas[1]], 
wave_directions=[betas[1]],
radiating_dofs=[rad_dofs[1]],
influenced_dofs=rad_dofs[1:3])

# Create DimStack of results
problems = problems_from_data(parameters, fb_array)
results = solve_all_problems(problems)
data = assemble_hydrodynamic_coefficients(parameters, fb_array, results)
# mhresults = compute_hydrodynamic_coefficients(parameters, fb_array)

In [ ]:
# Single dof differentiability

# Create FloatingBody struct
floatingbody1 = FloatingBody(mhmesh,
    [:Heave],
    [rotation_center...],
    "Horizontal Cylinder 1")

# Create parameters NamedTuple
parameters = (wave_frequencies=[omegas[1]], 
wave_directions=[betas[1]],
radiating_dofs=[:Heave])


function get_added_mass(omega)
    parameters = (wave_frequencies=[omega], 
    wave_directions=[betas[1]],
    radiating_dofs=[:Heave])
    mhresults = compute_hydrodynamic_coefficients(parameters, floatingbody1)
    added_mass = mhresults.added_mass[1]
    return added_mass
end


get_added_mass(1.5)

In [ ]:
using Zygote

In [ ]:

# A_w_grad, = Zygote.gradient(omega -> get_added_mass(omega),1.5)


In [ ]:
function get_added_mass2(omega)
    parameters = (wave_frequencies=[omega], 
    wave_directions=[betas[1]],
    radiating_dofs=[:Heave])
    mhresults = compute_and_label_hydrodynamic_coefficients(parameters, floatingbody1)
    added_mass = only(collect(mhresults.added_mass[wave_frequencies = At(omega)]))
    return added_mass
end
get_added_mass2(1.03)
# A_w_grad, = Zygote.gradient(omega -> get_added_mass2(omega),1.5)